# Facial Recognition in Historical Photographs with AI in Python

**Source lesson:** [Programming Historian — Facial Recognition in Historical Photographs with Artificial Intelligence in Python](https://programminghistorian.org/en/lessons/facial-recognition-ai-python) by Charles Goldberg and Zach Haala (CC-BY 4.0).

This notebook is my working version of that lesson: it walks through the same steps, but the code cells are **fill-in-the-blank** — I write the code myself, using the lesson's explanations as a guide.

**Run this in Google Colab.** It needs `dlib`, `TensorFlow`, a C++ compiler, and works best with GPU access — all of which Colab provides out of the box. Running locally would require setting all of that up manually.

**Dataset:** Yes — this lesson downloads its own dataset for you, no separate download needed on my part:
- A zip of six digitized Bethel University yearbooks (1911, 1921, 1931, 1941, 1951, 1961) as PDFs, pulled from Google Drive via `gdown` in the setup cell below.
- The OpenCV Haar Cascade face-detection model (`haarcascade_frontalface_default.xml`) ships inside the `opencv-python` package itself (accessed via `cv2.data.haarcascades`), so it doesn't need a separate download either.


## Learning Outcomes

By the end of this notebook I should be able to:
- Explain the basics of computer vision machine learning
- Explain how a computer "views" an image (pixels → RGB values)
- Extract pictures of human faces from historical documents digitized as PDFs, and organize them for further analysis
- Apply a pre-trained computer vision model (Haar Cascade, then DeepFace) to a historical dataset

## Prior Skills Assumed
- Basic Python 3 familiarity
- A Google account (for Colab)
- Basic familiarity with Jupyter/Colab notebooks


## A Note on Ethics Before I Start

This lesson explicitly warns that facial recognition and emotion-detection ML models are **not neutral tools**. Before running any code, it's worth sitting with why:

- Object recognition models can fail to detect or misidentify non-white and female faces more often than white male faces, partly because of how training data has historically been assembled.
- This specific dataset (Bethel University yearbooks) is majority white and male throughout the 20th century, which limits how representative any findings will be.
- Historical photos differ from the modern photos most models are trained on (poor quality, faded, rescanned), which can compound bias.

**Reflection question 1:** *Before running any code, what's one reason a facial-recognition model trained mostly on modern, high-quality photos of white faces might struggle with a 1911 yearbook scan? Write 2-3 sentences.*

> Your answer here: Probably both the overall picture quality also without color does not offer as many relevant tokens for the model to recognize and also the model is trained to recognize with those extra more relevant tokens. 
>
>


## Step 1: Lesson Setup

This first cell needs to:
1. Create a working folder called `yearbook` and move into it
2. Install `gdown` (a tool for downloading files from Google Drive) and use it to download the lesson's zipped dataset
3. Unzip the dataset and create an `images` subfolder
4. Install the ML/PDF libraries this lesson depends on: `PyMuPDF` (PDF → image conversion), `dlib` and `DeepFace` (face/emotion detection); OpenCV (`cv2`) comes bundled with `DeepFace`'s dependencies
5. Import everything needed and disable GPU use (Colab + TensorFlow compatibility workaround)

**TODO:** Fill in the blanks marked `# TODO`. The Google Drive file ID for the dataset is `1NHT8NN8ClBEnUC5VqkP3wr2KhyiIQzyU`.

In [1]:
%mkdir yearbook
%cd yearbook

!pip install --upgrade --no-cache-dir gdown

!gdown --id "1NHT8NN8ClBEnUC5VqkP3wr2KhyiIQzyU"

!unzip PHfiles.zip

%mkdir images

!pip install PyMuPDF
!pip install dlib
!pip install DeepFace

import os, shutil, fitz, cv2, numpy as np, pandas as pd, dlib, tensorflow as tf
from os.path import dirname, join
from deepface import DeepFace

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"


/Users/theowolf/Documents/GitHub/theojwolf.github.io/portfolio/facial-recognition-yearbooks/yearbook
zsh:1: command not found: pip
zsh:1: command not found: gdown
unzip:  cannot find or open PHfiles.zip, PHfiles.zip.zip or PHfiles.zip.ZIP.
zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip


ModuleNotFoundError: No module named 'fitz'

## Step 2: PDF Conversion

Now I'll convert each yearbook PDF into individual page images. For each PDF file (e.g. `1911.pdf`) this step should:

1. Find every file ending in `.pdf` in the current folder
2. Create a subfolder inside `/images` named after the PDF (e.g. `/images/1911`)
3. Copy the PDF into its new subfolder
4. Use PyMuPDF (`fitz`) to open the PDF and save each page as a separate `.png` image (e.g. `page-0.png`, `page-1.png`, ...)

**Hints:**
- `[f for f in os.listdir(path) if f.endswith('.pdf')]` gives you the list of PDF filenames
- `pdf.split(".")[0]` strips `.pdf` off a filename, leaving just the year
- `shutil.copy(src, dst)` copies a file
- `fitz.open(pdf)` opens a PDF; looping `for page in doc:` gives you each page
- `page.get_pixmap()` renders a page to an image object; `.save("page-%i.png" % page.number)` writes it to disk
- Use `os.chdir(...)` to move between folders as you go, and `os.chdir("..")` / `os.chdir(os.path.dirname(os.getcwd()))` to move back up

In [ ]:
path = r'./'

# TODO: build a list of all filenames in `path` that end with '.pdf'
pdfs = None

for pdf in pdfs:
    # TODO: move into ./images

    # TODO: create a new subfolder named after this pdf's year (strip the ".pdf")

    # TODO: build the path to that new subfolder and store it in `newdir`
    newdir = None

    # TODO: move back up one directory

    print("Now copying images into " + (newdir))
    # TODO: copy this pdf into `newdir`

    # TODO: move into `newdir`

    # TODO: open this pdf with fitz and store it in `doc`
    doc = None

    for page in doc:
        # TODO: render this page to a pixmap and save it as "page-<number>.png"
        pass

    # TODO: move back up two directories (out of the year folder, then out of /images)


## Step 3: Fundamentals of Object Detection (read + reflect, no code)

The lesson explains that:
- A computer sees an image as a grid of pixels, each represented by three numbers (R, G, B) from 0-255
- To save processing power, images are often **downscaled** and converted to **greyscale** before analysis ("thresholding")
- A **Haar Cascade** is a fast, older (2001) object-detection technique that looks for patterns of pixel groups rather than every individual pixel, using many "weak classifiers" that combine into a "strong classifier" when enough of them agree a face is present

**Reflection question 2:** *Why would greyscale conversion and downscaling make a Haar Cascade faster, at the cost of what kind of information?*

> Your answer here:
>
>

**Reflection question 3:** *The lesson notes that lighter skin tones tend to produce greater pixel-density differences than darker skin tones for a computer to detect. Given that Bethel University's yearbooks are majority white throughout this dataset, what effect might that have on the face-detection counts you get in Step 4, independent of any bias in the algorithm itself?*

> Your answer here:
>
>


## Step 4: Processing the Images (Face Detection)

Now I'll walk through every `.png` page image and use OpenCV's Haar Cascade to find and crop out individual faces, saving each one into a new `/<year> faces` folder.

For each year's folder, this step should:
1. List every `.png` file in that folder
2. If a `/<year> faces` subfolder doesn't already exist, create it
3. For each `.png` page image:
   - Read it in with OpenCV
   - Convert it to greyscale
   - Load OpenCV's pretrained frontal-face Haar Cascade (`cv2.data.haarcascades + "haarcascade_frontalface_default.xml"`)
   - Run `detectMultiScale` on the greyscale image to get bounding boxes `(x, y, w, h)` for each detected face
   - Crop each face out (with a little padding so the crop isn't too tight), draw a rectangle on it, and save it as a new `.png` in the `/<year> faces` folder
   - Delete the original full-page `.png` once its faces have been extracted

**Hints:**
- `cv2.imread(png)` reads an image; `cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)` converts it to greyscale
- `cv2.CascadeClassifier(...)` loads the Haar Cascade model
- `face_cascade.detectMultiScale(image=..., scaleFactor=1.9, minNeighbors=4)` returns a list of `(x, y, w, h)` boxes
- Slicing `image[y1:y2, x1:x2]` crops a NumPy image array
- Wrap the crop in a `try/except` — not every detected box will crop cleanly

In [ ]:
path = r'./'

# TODO: move into the images folder
dirs = os.listdir(path)

for dir in dirs:
    # TODO: move into this year's subfolder

    # TODO: build a list of every filename in this folder ending in '.png'
    pngs = None

    if not os.path.exists((dir) + ' faces'):
        print("New 'faces' directory created in " + (dir) + " folder")
        # TODO: create the "<year> faces" directory

        count = 0
        for png in pngs:
            # TODO: read the image in with cv2

            # TODO: convert it to greyscale

            # TODO: load the Haar Cascade frontal-face classifier

            # TODO: run detectMultiScale on the greyscale image (scaleFactor=1.9, minNeighbors=4)
            detected_faces = None

            count = 0
            for (x, y, w, h) in detected_faces:
                try:
                    xpadding = 20
                    ypadding = 40
                    # TODO: crop the image to this face's bounding box, plus padding
                    crop_face = None
                    count += 1
                    # TODO: draw a rectangle on the crop to show the detected bounding box

                    # TODO: write the cropped face out to '<year> faces/<count>_<original filename>.png'

                except (Exception):
                    print("An error happened")
                    continue
            # TODO: remove the original full-page png now that its faces are extracted
    # TODO: move back up out of this year's folder


## Step 5: Identifying Smiles

Haar Cascades are fast but rigid — they only recognize what they were explicitly trained to recognize. For emotion detection, this lesson switches to **DeepFace**, which uses a Convolutional Neural Network (CNN): a "deep learning" model made of layered "neurons" that each learn to recognize small patterns (an eye corner, an ear lobe, ...) and pass their findings to the next layer.

A photo is treated as containing a smile if DeepFace's dominant predicted emotion is `'happy'`; anything else counts as a non-smile.

This step should, for each year:
1. Walk through the `<year> faces` folder and collect every cropped face `.png`
2. Run `DeepFace.analyze(png, actions=['emotion'])` on each one
3. Tally how many are `'happy'` (smiles) vs. anything else (non-smiles), and how many raise an error
4. Convert each year's tallies into ratios (smiles / total, non-smiles / total, errors / total)
5. Save the results to a `YearbookOutput.csv` file with columns: `Years`, `Smiles`, `Non-Smiles`, `Error Weight`

**Reflection question 4:** *DeepFace's emotion classifier reduces every face to one of a handful of labeled emotions, with 'happy' as our stand-in for 'smiling.' What's a case where that mapping might misclassify a real historical photograph?*

> Your answer here:
>
>


In [ ]:
%cd ..
number_smiles = 0
smile_counts = []
number_nonsmiles = 0
nonsmile_counts = []
num_errors = 0
error_counts = []
pngs = []

file_count = 0
file_count_list = []
iterator = 0
count = 0
total_loops = 0

years = ['1911', '1921', '1931', '1941', '1951', '1961']

for year in years:
    path = r'./images' + '/' + year
    for root, dirs, files in os.walk(path):
        for _ in dirs:
            path = path + '/' + (year + ' faces')
            if (file_count != 0):
                file_count_list.append(file_count)
            file_count = 0
            for f in os.listdir(path):
                if f.endswith('.png'):
                    pngs.append(path + '/' + f)
                    file_count = file_count + 1

# TODO: loop over every path in `pngs` and, for each one:
#   - run DeepFace.analyze(png, actions=['emotion']) and print the result
#   - if demography[0]['dominant_emotion'] == 'happy', increment number_smiles, else number_nonsmiles
#   - when you've reached the end of a year's faces (count == file_count_list[iterator] + 1),
#     append the smile/non-smile/error ratios for that year to smile_counts / nonsmile_counts / error_counts,
#     then reset the counters and move to the next year (iterator += 1)
#   - wrap each DeepFace.analyze call in a try/except and count failures in num_errors
for png in pngs:
    pass

# TODO: build a pandas DataFrame with columns 'Years', 'Smiles', 'Non-Smiles', 'Error Weight'
# using the years list and the *_counts lists, then save it to 'YearbookOutput.csv' (no index column)


## Step 6: Download and Results

In Colab, download the finished CSV so I can open it in a spreadsheet app.

In [ ]:
# TODO: import `files` from google.colab and use files.download() to download 'YearbookOutput.csv'


### Interpreting My Results

The lesson's own run found smiling-face rates roughly like this (yours will vary run to run):

| Year | Smiles | Non-Smiles | Error Weight |
|------|--------|-----------|---------------|
| 1911 | 0%     | 100%      | 0%            |
| 1921 | 0%     | 92%       | 8%            |
| 1931 | 15%    | 70%       | 30% (interp.)  |
| 1941 | 3%     | 40%       | 53%           |
| 1951 | 25%    | 68%       | 14%           |
| 1961 | 43%    | 53%       | 9%            |

**Reflection question 5:** *How do your own `YearbookOutput.csv` numbers compare to the table above? If 1941 is an outlier in your run too, what did the lesson suggest as the likely explanation (hint: look at what kind of photos dominate that yearbook)?*

> Your answer here:
>
>

**Reflection question 6:** *The lesson connects the rising smile rate to broader social changes (e.g. the shift toward coeducation accelerating in the 1960s-70s). What's one alternative, non-social explanation for a rising smile rate over time that has nothing to do with who the students were (hint: think about camera/photography technology itself)?*

> Your answer here:
>
>


## Conclusions and Next Steps

Some directions to explore further, per the lesson:
- Training a custom object-recognition model instead of relying on pre-trained ones (see the Programming Historian lesson *Computer Vision for the Humanities: An Introduction to Deep Learning for Image Classification*)
- Applying the same "distant viewing" approach to a larger, more demographically diverse dataset
- Exploring other pre-trained models via Hugging Face, or point-and-click trainers like Roboflow

**Reflection question 7 (wrap-up):** *In 2-4 sentences, what did this exercise teach you about the tradeoff between how easy pre-trained ML models are to apply to historical images, and how trustworthy their output is as historical evidence?*

> Your answer here:
>
>
